In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,udf
from pyspark.sql.types import StringType,IntegerType
import time 

In [3]:
spark=SparkSession.builder.appName("advance_spark").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/13 10:46:08 WARN Utils: Your hostname, codespaces-d5e04d, resolves to a loopback address: 127.0.0.1; using 10.0.1.252 instead (on interface eth0)
26/06/13 10:46:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 10:46:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
df=spark.range(0,10000000).withColumn("value",col("id")%1000)


In [5]:
df.take(10)

[Row(id=0, value=0),
 Row(id=1, value=1),
 Row(id=2, value=2),
 Row(id=3, value=3),
 Row(id=4, value=4),
 Row(id=5, value=5),
 Row(id=6, value=6),
 Row(id=7, value=7),
 Row(id=8, value=8),
 Row(id=9, value=9)]

In [6]:
print("before part",df.rdd.getNumPartitions())

before part 2


In [7]:
df_repartition=df.repartition(10)

In [8]:
print("after partition ",df_repartition.rdd.getNumPartitions())

after partition  10


In [9]:
print(df_repartition)

DataFrame[id: bigint, value: bigint]


In [10]:
df_coalesce=df_repartition.coalesce(2)

In [12]:
optimized_df=df.filter(col("value")>500).filter(col("id")<5000000)

In [13]:
optimized_df.explain()

== Physical Plan ==
*(1) Project [id#0L, (id#0L % 1000) AS value#1L]
+- *(1) Filter (((id#0L % 1000) > 500) AND (id#0L < 5000000))
   +- *(1) Range (0, 10000000, step=1, splits=2)




In [14]:
print("after coalesced",df_coalesce.rdd.getNumPartitions())

after coalesced 2


In [15]:
optimized_df=df.filter(col("value")>500).filter(col("id")<5000000).select("id","value")

In [16]:
optimized_df.explain()

== Physical Plan ==
*(1) Project [id#0L, (id#0L % 1000) AS value#1L]
+- *(1) Filter (((id#0L % 1000) > 500) AND (id#0L < 5000000))
   +- *(1) Range (0, 10000000, step=1, splits=2)




In [21]:
start_time=time.time()
count_uncached =optimized_df.count()
end_time=time.time()
print(f"1.optimized Execution|count:{count_uncached}|time:{round(end_time-start_time,4)} seconds")


1.optimized Execution|count:2495000|time:0.2384 seconds


In [22]:
optimized_df.cache()

26/06/13 10:47:51 WARN CacheManager: Asked to cache already cached data.


DataFrame[id: bigint, value: bigint]

In [23]:
start_time=time.time()
count_first_cache =optimized_df.count()
end_time=time.time()
print(f"2.first cache action|count:{count_first_cache}|time:{round(end_time-start_time)} seconds")


2.first cache action|count:2495000|time:0 seconds


In [24]:
start_time=time.time()
count_second_cache =optimized_df.count()
end_time=time.time()
print(f"3.second cache action|count:{count_second_cache}|time:{round(end_time-start_time)} seconds")


3.second cache action|count:2495000|time:0 seconds


In [25]:
optimized_df.unpersist()

DataFrame[id: bigint, value: bigint]

In [26]:
data=[("Alice",25),("Bob",12),("Charlie",30)]
df1=spark.createDataFrame(data,["name","age"])

In [27]:
df1.show()

+-------+---+
|   name|age|
+-------+---+
|  Alice| 25|
|    Bob| 12|
|Charlie| 30|
+-------+---+



In [33]:
def categorize_age(age):
    if age>=18:
        return "Adult"
    elif 13< age<18:
     return "Teenager"
    else:
        return "Child"
    

In [34]:
age_category_udf=udf(categorize_age,StringType())

In [35]:
df_method=df1.withColumn("age_category",age_category_udf(col("age")))
print("method 1:standard udf via dataframe api")
df_method.show()

method 1:standard udf via dataframe api


+-------+---+------------+
|   name|age|age_category|
+-------+---+------------+
|  Alice| 25|       Adult|
|    Bob| 12|       Child|
|Charlie| 30|       Adult|
+-------+---+------------+



In [36]:
#for sql
spark.udf.register("sql_categorize_age",categorize_age,StringType())
df1.createOrReplaceTempView("people")

26/06/13 10:50:55 WARN SimpleFunctionRegistry: The function sql_categorize_age replaced a previously registered function.


In [37]:
sql_df=spark.sql("SELECT name,age,sql_categorize_age(age) as age_category FROM people")
sql_df.show()


+-------+---+------------+
|   name|age|age_category|
+-------+---+------------+
|  Alice| 25|       Adult|
|    Bob| 12|       Child|
|Charlie| 30|       Adult|
+-------+---+------------+

